# Indexing and slicing arrays

## Learning objectives

By the end of this notebook you will be able to:

- select single elements, rows and columns
- slice one- and two-dimensional arrays
- filter with boolean indexing and combined conditions
- use fancy indexing and np.where
- understand views versus copies

## Introduction

Selecting parts of an array is the basis of every analysis: one patient's results, all values above a
threshold, the columns for one condition.

In [1]:
import numpy as np
rng = np.random.default_rng(3)
glucose = rng.normal(6.2, 1.4, 12).round(1)
print(glucose)

[ 9.1  2.6  6.8  5.4  5.6  5.9  3.4  5.9  5.  10.9  6.5  5.7]


## One dimension

Exactly like lists: positions from 0, negatives from the end, slices exclude the stop.

In [2]:
print(glucose[0], glucose[-1])
print(glucose[2:5], glucose[:3], glucose[::3])

9.1 5.7
[6.8 5.4 5.6] [9.1 2.6 6.8] [ 9.1  5.4  3.4 10.9]


## Two dimensions

`arr[row, column]`. A colon `:` on its own means "all".

A table of 6 participants (rows) by 4 visits (columns):

In [3]:
sbp = np.array([
    [142, 138, 135, 131],
    [128, 130, 127, 126],
    [155, 150, 146, 141],
    [118, 121, 119, 117],
    [149, 147, 140, 138],
    [133, 129, 128, 124],
])
print(sbp[2, 0])         # participant 3, visit 1
print(sbp[0])            # participant 1, all visits
print(sbp[:, 0])         # all participants, visit 1
print(sbp[:, -1])        # last visit
print(sbp[1:3, 1:3])     # a block

155
[142 138 135 131]
[142 128 155 118 149 133]
[131 126 141 117 138 124]
[[130 127]
 [150 146]]


Selecting a single column gives a 1-D array. Keep the result 2-D with a slice (`sbp[:, 0:1]`).

In [4]:
print(sbp[:, 0].shape, sbp[:, 0:1].shape)

(6,) (6, 1)


## Boolean indexing

A comparison gives an array of True/False values (a *mask*). Using the mask as an index keeps the True
positions.

In [5]:
mask = glucose >= 7.0
print(mask)
print(glucose[mask])
print(glucose[glucose < 5.6])

[ True False False False False False False False False  True False False]
[ 9.1 10.9]
[2.6 5.4 3.4 5. ]


Counting and proportions: True counts as 1.

In [6]:
print(mask.sum(), "values at or above 7.0")
print(f"{mask.mean():.0%} of values")

2 values at or above 7.0
17% of values


### Combining conditions

Use `&` (and), `|` (or), `~` (not), with **brackets around each condition**. Python's `and`/`or` do
not work on arrays.

In [7]:
print(glucose[(glucose >= 5.6) & (glucose < 7.0)])       # pre-diabetic range
print(glucose[(glucose < 4.0) | (glucose > 8.0)])        # outside a range
print(glucose[~(glucose >= 7.0)])
try:
    glucose[glucose > 5 and glucose < 7]
except ValueError as e:
    print("ValueError:", str(e)[:60], "...")

[6.8 5.6 5.9 5.9 6.5 5.7]
[ 9.1  2.6  3.4 10.9]
[2.6 6.8 5.4 5.6 5.9 3.4 5.9 5.  6.5 5.7]
ValueError: The truth value of an array with more than one element is am ...


### Boolean indexing on a table

Participants whose first visit was 140 or above, and their whole rows:

In [8]:
high_at_start = sbp[:, 0] >= 140
print(high_at_start)
print(sbp[high_at_start])
print("change over follow-up:", sbp[high_at_start, -1] - sbp[high_at_start, 0])

[ True False  True False  True False]
[[142 138 135 131]
 [155 150 146 141]
 [149 147 140 138]]
change over follow-up: [-11 -14 -11]


## Fancy indexing

An array or list of positions selects those items, in that order.

In [9]:
print(glucose[[0, 3, 7]])
print(sbp[[0, 2, 4], :][:, [0, 3]])     # rows 0, 2, 4; first and last visits
order = np.argsort(glucose)
print(order)
print(glucose[order])                   # sorted values

[9.1 5.4 5.9]
[[142 131]
 [155 141]
 [149 138]]
[ 1  6  8  3  4 11  5  7 10  2  0  9]
[ 2.6  3.4  5.   5.4  5.6  5.7  5.9  5.9  6.5  6.8  9.1 10.9]


`np.argsort` returns the *positions* that would sort the array, which lets you sort one array by
another.

## np.where

`np.where(condition)` gives the positions where a condition is true;
`np.where(condition, a, b)` chooses between two values element by element.

In [10]:
print(np.where(glucose >= 7.0))
labels = np.where(glucose >= 7.0, "high", "normal")
print(labels)
capped = np.where(glucose > 8, 8, glucose)
print(capped)

(array([0, 9]),)
['high' 'normal' 'normal' 'normal' 'normal' 'normal' 'normal' 'normal'
 'normal' 'high' 'normal' 'normal']
[8.  2.6 6.8 5.4 5.6 5.9 3.4 5.9 5.  8.  6.5 5.7]


`np.select` handles more than two categories:

In [11]:
category = np.select([glucose < 5.6, glucose < 7.0], ["normal", "borderline"], default="high")
print(list(zip(glucose, category)))

[(np.float64(9.1), np.str_('high')), (np.float64(2.6), np.str_('normal')), (np.float64(6.8), np.str_('borderline')), (np.float64(5.4), np.str_('normal')), (np.float64(5.6), np.str_('borderline')), (np.float64(5.9), np.str_('borderline')), (np.float64(3.4), np.str_('normal')), (np.float64(5.9), np.str_('borderline')), (np.float64(5.0), np.str_('normal')), (np.float64(10.9), np.str_('high')), (np.float64(6.5), np.str_('borderline')), (np.float64(5.7), np.str_('borderline'))]


## Changing values through an index

In [12]:
readings = np.array([5.1, -1.0, 6.4, 999.0, 7.2])
readings[(readings < 0) | (readings > 100)] = np.nan      # mark impossible values as missing
print(readings)

[5.1 nan 6.4 nan 7.2]


## Views and copies

A **slice** is a *view*: it shares data with the original, so changing it changes the original.
Boolean and fancy indexing return **copies**. Use `.copy()` when you want an independent array.

In [13]:
a = np.arange(6)
v = a[1:4]
v[0] = 100
print(a)                 # changed through the view

b = np.arange(6)
c = b[1:4].copy()
c[0] = 100
print(b)                 # unchanged

[  0 100   2   3   4   5]
[0 1 2 3 4 5]


## Practise

Exercises for this folder are in `exercises/NumPy_Exercises.ipynb`.

## Further reading

- [NumPy user guide](https://numpy.org/doc/stable/user/index.html)
- [Indexing on ndarrays](https://numpy.org/doc/stable/user/basics.indexing.html)
- [np.where](https://numpy.org/doc/stable/reference/generated/numpy.where.html)